In [17]:
import pandas as pd
import numpy as np

# Center Point: Sector 75, Noida
center_lat = 28.5760
center_lon = 77.3823

# Create a smaller grid to stay under Mappls API limits
grid_size = 14 # 14x14 = 196 points

lats = np.linspace(center_lat - 0.05, center_lat + 0.05, grid_size)
lons = np.linspace(center_lon - 0.05, center_lon + 0.05, grid_size)

destinations = []
for lat in lats:
    for lon in lons:
        destinations.append({"lat": round(lat, 6), "lon": round(lon, 6)})

df_destinations = pd.DataFrame(destinations)
df_destinations.to_csv("noida_nodes.csv", index=False)
print(f"Generated {len(df_destinations)} nodes and saved to noida_nodes.csv")

Generated 196 nodes and saved to noida_nodes.csv


In [21]:
import requests
import pandas as pd
import time
import os

# PASTE YOUR MAPPLS REST API KEY HERE
MAPPLS_REST_KEY = "###############################"

ORIGIN_LAT = 28.5760
ORIGIN_LON = 77.3823

df_nodes = pd.read_csv("noida_nodes.csv") 

print(f"Starting Mappls Route ADV API Extraction for {len(df_nodes)} nodes...")

# Remove the old CSV if it exists so we start fresh
if os.path.exists("travel_times_results.csv"):
    os.remove("travel_times_results.csv")

for index, row in df_nodes.iterrows():
    # FAILSAFE: Stop at 195 to guarantee we do not exhaust the 200 limit
    if index >= 195:
        print("Approaching API limit! Failsafe triggered. Stopping extraction.")
        break

    dest_lat = row['lat']
    dest_lon = row['lon']
    
    # NEW ENDPOINT: Route ADV API
    coordinates = f"{ORIGIN_LON},{ORIGIN_LAT};{dest_lon},{dest_lat}"
    url = f"https://apis.mappls.com/advancedmaps/v1/{MAPPLS_REST_KEY}/route_adv/driving/{coordinates}"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        # FIXED: Mappls Route ADV uses "code": "Ok" instead of responseCode: 200
        if data.get("code") == "Ok" and "routes" in data:
            duration_mins = data["routes"][0]["duration"] / 60
            distance_km = data["routes"][0]["distance"] / 1000
            
            temp_df = pd.DataFrame([{
                "lat": dest_lat,
                "lon": dest_lon,
                "distance_km": round(distance_km, 2),
                "duration_mins": round(duration_mins, 2)
            }])
            
            # Save line-by-line
            temp_df.to_csv("travel_times_results.csv", mode='a', header=not os.path.exists("travel_times_results.csv"), index=False)
            
        else:
            print(f"API Error at node {index}. Response: {data}")
            
    except Exception as e:
        print(f"Request failed for node {index}: {e}")
        
    time.sleep(1)
    
    if (index + 1) % 10 == 0:
        print(f"Processed node {index + 1}/{len(df_nodes)}")

print("Extraction complete! Everything is safely saved in travel_times_results.csv")

Starting Mappls Route ADV API Extraction for 196 nodes...
Processed node 10/196
Processed node 20/196
Processed node 30/196
Processed node 40/196
Processed node 50/196
Processed node 60/196
Processed node 70/196
Processed node 80/196
Processed node 90/196
Processed node 100/196
Processed node 110/196
Processed node 120/196
Processed node 130/196
Processed node 140/196
Processed node 150/196
Processed node 160/196
Processed node 170/196
Processed node 180/196
Processed node 190/196
Approaching API limit! Failsafe triggered. Stopping extraction.
Extraction complete! Everything is safely saved in travel_times_results.csv


In [29]:
import pandas as pd
import folium
from folium import plugins

# 1. Load the successfully extracted data
df = pd.read_csv("travel_times_results.csv")

# Filter out any failed routing requests just in case
df = df[df['duration_mins'] > 0]
print(f"Total valid MapmyIndia routes plotted: {len(df)} points.")

# 2. Setup the Dark Mode Map
# Center it exactly on your home base in Sector 75, Noida
ORIGIN_LAT = 28.5760
ORIGIN_LON = 77.3823
m = folium.Map(location=[ORIGIN_LAT, ORIGIN_LON], zoom_start=13, tiles="CartoDB positron")

# Add a prominent marker for your Origin point
folium.Marker(
    [ORIGIN_LAT, ORIGIN_LON],
    popup="<b>Origin: Sector 75</b>",
    icon=folium.Icon(color="white", icon="home", prefix='fa')
).add_to(m)

# 3. Prepare data for the HeatMap
# The HeatMap plugin expects a list of [latitude, longitude, weight]
# We use the travel duration as the "weight" (higher time = hotter color)
heat_data = [[row['lat'], row['lon'], row['duration_mins']] for index, row in df.iterrows()]

# 4. Add the HeatMap Layer
# Radius and blur are kept wide so the 195 points blend together over the roads
plugins.HeatMap(
    heat_data,
    name="Travel Time Heatmap",
    radius=25,          
    blur=15,            
    max_zoom=1, 
    min_opacity=0.2,
    gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1.0: 'red'}
).add_to(m)

# 5. Add small interactive dots (so you can still hover over them for exact times)
for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3, 
        color="black",
        weight=0.5,
        fill=True,
        fill_opacity=0.2,
        tooltip=f"Distance: {row['distance_km']} km<br>Time: {row['duration_mins']} mins" 
    ).add_to(m)

# 6. Save and output
output_file = "noida_mappls_heatmap.html"
m.save(output_file)
print(f"Success! Open {output_file} in your browser to see your upgraded map.")

Total valid MapmyIndia routes plotted: 195 points.
Success! Open noida_mappls_heatmap.html in your browser to see your upgraded map.
